In [1]:
!nvidia-smi

Wed Feb 26 20:47:44 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.47                 Driver Version: 572.47         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090      WDDM  |   00000000:01:00.0  On |                  Off |
|  0%   35C    P8             24W /  450W |    1535MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
import numpy as np
torch.manual_seed(7)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
EPOCHS = 20
BATCH_SIZE = 1

In [3]:
device

device(type='cuda', index=0)

In [4]:
# Load training dataset into a single batch to compute mean and stddev.
transform = transforms.Compose([transforms.ToTensor()])
trainset = MNIST(root='./pt_data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=len(trainset), shuffle=True)
data = next(iter(trainloader))
mean = data[0].mean()
stddev = data[0].std()

# Helper function needed to standardize data when loading datasets.
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize(mean, stddev)])

trainset = MNIST(root='./pt_data', train=True, download=True, transform=transform)
testset = MNIST(root='./pt_data', train=False, download=True, transform=transform)

In [5]:
trainset

Dataset MNIST
    Number of datapoints: 60000
    Root location: ./pt_data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.13066047430038452, std=0.30810782313346863)
           )

In [6]:
testset

Dataset MNIST
    Number of datapoints: 10000
    Root location: ./pt_data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=0.13066047430038452, std=0.30810782313346863)
           )

In [7]:
# Create a Sequential (feed-forward) model.
# 784 inputs.
# Two fully-connected layers with 25 and 10 neurons.
# tanh as activation function for hidden layer.
# Logistic (sigmoid) as activation function for output layer.
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 25),
    nn.Tanh(),
    nn.Linear(25, 10),
    nn.Sigmoid()
)

# Initialize weights.
for module in model.modules():
    if isinstance(module, nn.Linear):
        nn.init.uniform_(module.weight, a=-0.1, b=0.1)
        nn.init.constant_(module.bias, 0.0)

In [8]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

In [9]:
loss_function = nn.MSELoss()

# Transfer model to GPU
model.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=25, bias=True)
  (2): Tanh()
  (3): Linear(in_features=25, out_features=10, bias=True)
  (4): Sigmoid()
)

In [10]:
# Create DataLoader objects that will help create mini-batches.
trainloader = DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True)
testloader = DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# Train the model. In PyTorch we have to implement the training loop ourselves.
for i in range(EPOCHS):
    model.train() # Set model in training mode.
    train_loss = 0.0
    train_correct = 0
    train_batches = 0
    for inputs, targets in trainloader:
        # Move data to GPU.
        one_hot_targets = nn.functional.one_hot(targets, num_classes=10).float()
        inputs, targets, one_hot_targets = inputs.to(device), targets.to(device), one_hot_targets.to(device)

        # Zero the parameter gradients.
        optimizer.zero_grad()

        # Forward pass.
        outputs = model(inputs)
        loss = loss_function(outputs, one_hot_targets)

        # Accumulate metrics.
        _, indices = torch.max(outputs.data, 1)
        train_correct += (indices == targets).sum().item()
        train_batches +=  1
        train_loss += loss.item()

        # Backward pass and update.
        loss.backward()
        optimizer.step()

    train_loss = train_loss / train_batches
    train_acc = train_correct / (train_batches * BATCH_SIZE)

    # Evaluate the model on the test dataset. Identical to loop above but without
    # weight adjustment.
    model.eval() # Set model in inference mode.
    test_loss = 0.0
    test_correct = 0
    test_batches = 0
    for inputs, targets in testloader:
        one_hot_targets = nn.functional.one_hot(targets, num_classes=10).float()
        inputs, targets, one_hot_targets = inputs.to(device), targets.to(device), one_hot_targets.to(device)
        outputs = model(inputs)
        loss = loss_function(outputs, one_hot_targets)
        _, indices = torch.max(outputs, 1)
        test_correct += (indices == targets).sum().item()
        test_batches +=  1
        test_loss += loss.item()

    test_loss = test_loss / test_batches
    test_acc = test_correct / (test_batches * BATCH_SIZE)

    print(f'Epoch {i+1}/{EPOCHS} loss: {train_loss:.4f} - acc: {train_acc:0.4f} - val_loss: {test_loss:.4f} - val_acc: {test_acc:0.4f}')
    

Epoch 1/20 loss: 0.0501 - acc: 0.7087 - val_loss: 0.0238 - val_acc: 0.8955
Epoch 2/20 loss: 0.0198 - acc: 0.9030 - val_loss: 0.0161 - val_acc: 0.9176
Epoch 3/20 loss: 0.0156 - acc: 0.9163 - val_loss: 0.0140 - val_acc: 0.9231
Epoch 4/20 loss: 0.0139 - acc: 0.9227 - val_loss: 0.0129 - val_acc: 0.9276
Epoch 5/20 loss: 0.0130 - acc: 0.9268 - val_loss: 0.0122 - val_acc: 0.9302
Epoch 6/20 loss: 0.0122 - acc: 0.9312 - val_loss: 0.0118 - val_acc: 0.9309
Epoch 7/20 loss: 0.0116 - acc: 0.9338 - val_loss: 0.0115 - val_acc: 0.9339
Epoch 8/20 loss: 0.0112 - acc: 0.9371 - val_loss: 0.0112 - val_acc: 0.9352
Epoch 9/20 loss: 0.0107 - acc: 0.9397 - val_loss: 0.0109 - val_acc: 0.9354
Epoch 10/20 loss: 0.0104 - acc: 0.9410 - val_loss: 0.0106 - val_acc: 0.9374
Epoch 11/20 loss: 0.0101 - acc: 0.9434 - val_loss: 0.0104 - val_acc: 0.9394
Epoch 12/20 loss: 0.0098 - acc: 0.9449 - val_loss: 0.0103 - val_acc: 0.9400
Epoch 13/20 loss: 0.0096 - acc: 0.9466 - val_loss: 0.0101 - val_acc: 0.9398
Epoch 14/20 loss: 0.0